# Mitterrand 1981 — Agent conversationnel

Point d'entrée unique du projet : préparation des données, QLoRA sur `google/gemma-4-12B-it`, comparaison avant/après et interface Gradio.

Deux parcours, depuis un environnement neuf :
- **Entraîner et discuter :** mettre `RUN_TRAINING = True` et exécuter les cellules dans l'ordre.
- **Discuter avec l’adaptateur sur Drive :** garder `RUN_TRAINING = False` et `ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"`, puis exécuter les mêmes cellules. Les cellules d'entraînement sont ignorées.

**Environnement :** Colab avec GPU L4 ou A100 pour l'entraînement, accès Hugging Face au modèle et jeton `HF_TOKEN` (ou connexion interactive). L'adaptateur peut être sauvegardé sur Drive ou publié sur Hugging Face. Un T4 suffit en mémoire pour discuter, mais le float16 peut être instable avec Gemma.

**Interface :** ouvrir Colab dans Chrome ou Firefox. L'interface s'affiche sous la dernière cellule, sans lien public. L'extension Colab de VS Code ne transfère pas les ports Gradio : ouvrir le notebook dans Colab pour la démonstration.

Les cinq questions de comparaison déjà présentes sont conservées ; l'évaluation complète de la section 6 du sujet reste à ajouter. Ce notebook doit être exécuté avant la remise.


## 1. Installation


In [ ]:
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets ipywidgets matplotlib "gradio>=6,<7"


## 2. Configuration

Les hyperparamètres sont prévus pour ~550 exemples : 3 époques, soit environ 200 pas d'optimisation avec un batch effectif de 8. Le meilleur checkpoint (plus faible perte de validation) est rechargé à la fin, ce qui protège du sur-apprentissage.

In [ ]:
RUN_TRAINING = False  # @param {type:"boolean"}
ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"  # @param {type:"string"}

MODEL_ID = "google/gemma-4-12B-it"  # @param {type:"string"}
OUTPUT_DIR = "gemma-4-12b-mitterrand-qlora"  # @param {type:"string"}

MAX_LENGTH = 1024  # @param {type:"integer"}  les exemples font au plus ~800 tokens
NUM_EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
EFFECTIVE_BATCH_SIZE = 8  # @param {type:"integer"}
LORA_R = 16  # @param {type:"integer"}
LORA_ALPHA = 32  # @param {type:"integer"}
LORA_DROPOUT = 0.05  # @param {type:"number"}

SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/mitterrand-qlora"  # @param {type:"string"}

### Fichiers du projet

En local, lancer le notebook depuis le dépôt : les données, le prompt et l'app sont utilisés directement. Dans Colab, cette cellule récupère le dépôt. `REPO_BRANCH` doit désigner la branche contenant les fichiers du projet.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Kodaiii/nlp-mitterand.git"  # @param {type:"string"}
REPO_BRANCH = "dataset-qlora-mitterrand"  # @param {type:"string"}  mettre "main" une fois la PR fusionnée

_local_roots = [Path.cwd(), *Path.cwd().parents]
REPO_DIR = next((p for p in _local_roots if (p / "app" / "app.py").is_file() and (p / "data" / "system_prompt.txt").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/nlp-mitterand")
    if not (REPO_DIR / "app" / "app.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
print(f"Dépôt : {REPO_DIR}")


## 3. Connexion à Hugging Face

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    login(token=userdata.get("HF_TOKEN"))
    print("Connecté avec le secret HF_TOKEN.")
except Exception as error:  # secret absent, ou extension VS Code (userdata n'y est pas disponible)
    print(f"Secret HF_TOKEN indisponible ({error}), connexion interactive :")
    from huggingface_hub import notebook_login

    notebook_login()

## 4. Données

Chaque exemple est au format `{"messages": [system, user, assistant, ...]}`. Le prompt système est le même partout ; on le récupère dans les données pour le réutiliser à l'inférence.

In [ ]:
if RUN_TRAINING:
    import os

    import ipywidgets as widgets
    from IPython.display import display

    DATA_FILES = {split: str(REPO_DIR / "data" / name) for split, name in {"train": "mitterrand_train.jsonl", "validation": "mitterrand_val.jsonl"}.items()}
    missing = [name for name in DATA_FILES.values() if not os.path.exists(name)]

    # Widget de téléversement : il marche dans Colab comme dans VS Code (files.upload() n'existe pas dans l'extension).
    uploader = widgets.FileUpload(accept=".jsonl", multiple=True, description="Fichiers .jsonl")
    if missing:
        print(f"Fichiers manquants : {missing}.\nSélectionnez-les avec le bouton ci-dessous, puis exécutez la cellule suivante.")
        display(uploader)
    else:
        print("Fichiers de données déjà présents.")

In [ ]:
if RUN_TRAINING:
    def uploaded_files(value):
        # ipywidgets 7 (Colab) renvoie {nom: {"content": ...}}, ipywidgets 8 un tuple de dicts avec une clé "name".
        if isinstance(value, dict):
            return [(name, item["content"]) for name, item in value.items()]
        return [(item["name"], item["content"]) for item in value]


    for name, content in uploaded_files(uploader.value):
        with open(REPO_DIR / "data" / Path(name).name, "wb") as f:
            f.write(bytes(content))
        print(f"Enregistré : {name}")

    missing = [name for name in DATA_FILES.values() if not os.path.exists(name)]
    assert not missing, f"Toujours manquants : {missing}. Téléversez-les avec le bouton de la cellule précédente."

    from datasets import load_dataset

    dataset = load_dataset("json", data_files=DATA_FILES)
    print(dataset)

    SYSTEM_PROMPT = dataset["train"][0]["messages"][0]["content"]
    assert dataset["train"][0]["messages"][0]["role"] == "system", "Les données doivent commencer par un message system."
    print("\nPrompt système :\n" + SYSTEM_PROMPT)

## 5. Chargement du modèle en 4 bits

Quantification NF4 avec double quantification, calcul en bfloat16 (configuration du guide QLoRA officiel de Gemma 4).

In [ ]:
if RUN_TRAINING:
    import torch
    from peft import prepare_model_for_kbit_training
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

    gpu = torch.cuda.get_device_properties(0)
    GPU_MEMORY_GB = gpu.total_memory / 1024**3
    print(f"GPU : {gpu.name} ({GPU_MEMORY_GB:.0f} Go)")

    if torch.cuda.is_bf16_supported():
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16
        print("⚠️  Pas de bfloat16 sur ce GPU : l'entraînement en float16 peut être instable avec Gemma. Préférez un L4 ou un A100.")

    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype=torch_dtype,
        device_map="auto",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype,
            bnb_4bit_quant_storage=torch_dtype,
        ),
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = "right"

    # Recommandé pour QLoRA ; le guide officiel ne le saute que sur les GPU de 16 Go par manque de mémoire.
    if GPU_MEMORY_GB > 16:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.1f} Go")

## 6. Chat template et longueur des exemples

Le template de Gemma 4 découpe la conversation en tours `<|turn>system`, `<|turn>user` et `<|turn>model`, chacun fermé par `<turn|>`.

Sur le 12B (comme sur le 26B et le 31B), quand le mode « thinking » est désactivé, le template ouvre chaque réponse par un **canal de pensée vide** (`<|channel>thought\n<channel|>`). Google recommande de reproduire ce canal à l'entraînement quand les données n'ont pas de raisonnement explicite. On détecte donc ce que le template ajoute au moment de générer, et on l'insère au début de chaque tour du modèle dans les exemples d'entraînement. Ainsi, entraînement et inférence voient exactement le même format.

In [ ]:
if RUN_TRAINING:
    import re

    tokenizer = processor.tokenizer
    PROCESSOR_ADDS_BOS = processor(text="test", return_tensors="pt")["input_ids"][0][0].item() == tokenizer.bos_token_id
    MODEL_TURN_OPEN = "<|turn>model\n"

    # Ce que le template place après « <|turn>model\n » quand on lui demande de générer.
    _probe = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Bonjour"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    assert MODEL_TURN_OPEN in _probe, f"Format inattendu, vérifiez le template :\n{_probe}"
    GENERATION_PREFIX = _probe[_probe.rindex(MODEL_TURN_OPEN) + len(MODEL_TURN_OPEN) :]
    print(f"Préfixe de génération après {MODEL_TURN_OPEN!r} : {GENERATION_PREFIX!r}")


    def render(messages):
        # Texte au format Gemma 4, avec le même préfixe de génération au début de chaque tour du modèle.
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        if GENERATION_PREFIX:
            not_followed_by_prefix = re.escape(MODEL_TURN_OPEN) + f"(?!{re.escape(GENERATION_PREFIX)})"
            text = re.sub(not_followed_by_prefix, lambda _: MODEL_TURN_OPEN + GENERATION_PREFIX, text)
        text = text.strip()
        # On retire le <bos> du template si le processor en ajoute déjà un, pour éviter de le doubler.
        if PROCESSOR_ADDS_BOS and tokenizer.bos_token and text.startswith(tokenizer.bos_token):
            text = text[len(tokenizer.bos_token) :]
        return text


    print(render(dataset["train"][0]["messages"]))

    lengths = [len(processor(text=render(ex["messages"]))["input_ids"][0]) for split in dataset for ex in dataset[split]]
    too_long = sum(n > MAX_LENGTH for n in lengths)
    print(f"\nTokens par exemple : max {max(lengths)}, moyenne {sum(lengths) / len(lengths):.0f}")
    print(f"Exemples tronqués à {MAX_LENGTH} tokens : {too_long}" + ("  ⚠️ augmentez MAX_LENGTH" if too_long else ""))

## 7. Masquage de la perte : n'apprendre que les réponses de Mitterrand

La perte n'est calculée que sur le contenu des tours `model` et sur le `<turn|>` qui les ferme, pour que le modèle apprenne à s'arrêter. Tout le reste est masqué (`-100`) : le prompt système, les questions et l'en-tête du tour avec son canal de pensée vide, puisque ce dernier est fourni par le template à l'inférence. Contrairement au guide officiel, qui ne démasque que le premier tour `model`, on démasque **chaque** tour, ce qui compte pour les 35 dialogues multi-tours.

In [ ]:
if RUN_TRAINING:
    END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<turn|>")


    def find_subsequence(sequence, pattern, start=0):
        for index in range(start, len(sequence) - len(pattern) + 1):
            if sequence[index : index + len(pattern)] == pattern:
                return index
        return -1


    # En-tête d'un tour du modèle (« <|turn>model\n » + préfixe de génération), tel que le tokenizer le découpe dans un vrai exemple.
    _sample_ids = processor(text=render(dataset["train"][0]["messages"]))["input_ids"][0].tolist()
    _prefix_ids = tokenizer.encode(GENERATION_PREFIX, add_special_tokens=False) if GENERATION_PREFIX else []
    _candidates = [
        tokenizer.encode(MODEL_TURN_OPEN + GENERATION_PREFIX, add_special_tokens=False),
        [tokenizer.convert_tokens_to_ids(token) for token in ("<|turn>", "model", "\n")] + _prefix_ids,
    ]
    MODEL_HEADER_IDS = next((c for c in _candidates if find_subsequence(_sample_ids, c) >= 0), None)
    assert MODEL_HEADER_IDS is not None, "En-tête du tour « model » introuvable : vérifiez le rendu du template ci-dessus."


    def model_turn_labels(input_ids):
        # Labels = input_ids sur les tours du modèle (fermeture <turn|> incluse), -100 partout ailleurs.
        labels = [-100] * len(input_ids)
        position = find_subsequence(input_ids, MODEL_HEADER_IDS)
        while position >= 0:
            index = position + len(MODEL_HEADER_IDS)
            while index < len(input_ids) and input_ids[index] != END_OF_TURN_ID:
                labels[index] = input_ids[index]
                index += 1
            if index < len(input_ids):
                labels[index] = input_ids[index]
            position = find_subsequence(input_ids, MODEL_HEADER_IDS, index)
        return labels


    def collate_fn(examples):
        texts = [render(example["messages"]) for example in examples]
        batch = processor(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        labels = torch.tensor([model_turn_labels(ids) for ids in batch["input_ids"].tolist()])
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch


    # Contrôle visuel sur un dialogue multi-tours : en vert ce qui est appris, en gris ce qui est masqué.
    multi_turn = next(ex for ex in dataset["train"] if len(ex["messages"]) > 3)
    check = collate_fn([multi_turn])
    pieces = []
    for token_id, label in zip(check["input_ids"][0].tolist(), check["labels"][0].tolist()):
        piece = tokenizer.decode([token_id])
        pieces.append(f"\033[92m{piece}\033[0m" if label != -100 else f"\033[90m{piece}\033[0m")
    print("".join(pieces))
    print(f"\nTokens appris : {(check['labels'] != -100).sum().item()} / {check['labels'].numel()}")

### Fonction de génération commune

La même fonction sert à la baseline, au modèle adapté et à Gradio. Le contexte transmis par Gradio reste limité aux cinq derniers échanges.


In [ ]:
def ask(question, history=None, max_new_tokens=400):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + (history or []) + [{"role": "user", "content": question}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.95)
    return processor.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True).strip()



## 8. Avant l'entraînement : réponses du modèle d'origine

Ces réponses servent de point de comparaison. On pose les mêmes questions après l'entraînement.

In [ ]:
if RUN_TRAINING:
    TEST_QUESTIONS = [
        "Que proposez-vous pour les retraites ?",
        "Monsieur Mitterrand, pourquoi voudriez-vous abolir la peine de mort alors que les Français y sont favorables ?",
        "tu penses quoi de TikTok ?",
        "Pourquoi voter pour vous plutôt que pour Giscard ?",
        "Que pensez-vous de la force de dissuasion nucléaire ?",
    ]




    model.config.use_cache = True
    baseline_answers = {}
    for question in TEST_QUESTIONS:
        baseline_answers[question] = ask(question)
        print(f"❓ {question}\n\n{baseline_answers[question]}\n\n{'─' * 80}\n")

## 9. Configuration LoRA et de l'entraînement

Sans `target_modules`, PEFT applique ses valeurs par défaut pour Gemma 4, limitées aux couches du modèle de langage. Les encodeurs vision et audio restent gelés.

In [ ]:
if RUN_TRAINING:
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )

    per_device_batch_size = 2 if GPU_MEMORY_GB >= 40 else 1

    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        gradient_accumulation_steps=max(1, EFFECTIVE_BATCH_SIZE // per_device_batch_size),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch_fused",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_grad_norm=0.3,
        bf16=torch_dtype == torch.bfloat16,
        fp16=torch_dtype == torch.float16,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
        dataset_kwargs={"skip_prepare_dataset": True},  # le collator s'occupe du rendu et de la tokenisation
        remove_unused_columns=False,
    )

    model.config.use_cache = False  # incompatible avec le gradient checkpointing
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=peft_config,
        processing_class=processor,
        data_collator=collate_fn,
    )
    trainer.model.print_trainable_parameters()

## 10. Entraînement

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result.metrics)

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt

    history = trainer.state.log_history
    train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
    eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

    plt.figure(figsize=(8, 4))
    plt.plot(*zip(*train_points), label="perte d'entraînement")
    plt.plot(*zip(*eval_points), "o-", label="perte de validation")
    plt.xlabel("pas")
    plt.ylabel("perte")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    print(f"Meilleur checkpoint : {trainer.state.best_model_checkpoint} (eval_loss {trainer.state.best_metric:.3f})")

Si la perte de validation remonte nettement dès la 2ᵉ époque alors que celle d'entraînement continue de baisser, le modèle apprend les réponses par cœur. Le meilleur checkpoint est déjà rechargé, mais pour le prochain essai vous pouvez baisser `NUM_EPOCHS` à 2 ou `LEARNING_RATE` à `1e-4`.

## 11. Sauvegarde de l'adaptateur

Seul l'adaptateur LoRA est sauvegardé (quelques centaines de Mo). À l'utilisation, il se recharge par-dessus `google/gemma-4-12B-it`. Le prompt système est enregistré avec lui (`system_prompt.txt`), pour que le parcours de rechargement utilise exactement le même. La copie sur Google Drive fonctionne aussi depuis VS Code (extension v0.2.1 ou plus récente).

In [ ]:
if RUN_TRAINING:
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    with open(os.path.join(OUTPUT_DIR, "system_prompt.txt"), "w", encoding="utf-8") as f:
        f.write(SYSTEM_PROMPT)
    print(f"Adaptateur sauvegardé dans {OUTPUT_DIR}/")

    if SAVE_TO_DRIVE:
        import shutil

        from google.colab import drive

        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
        print(f"Copié dans Google Drive : {DRIVE_DIR}")

## 12. Après l'entraînement : mêmes questions

In [ ]:
if RUN_TRAINING:
    model = trainer.model
    model.train(False)  # mode inférence : désactive le dropout LoRA
    model.config.use_cache = True

    for question in TEST_QUESTIONS:
        print(f"❓ {question}\n")
        print(f"AVANT :\n{baseline_answers[question]}\n")
        print(f"APRÈS :\n{ask(question)}\n\n{'─' * 80}\n")

Comparaison avec les réponses de référence sur des exemples de validation, jamais vus pendant l'entraînement :

In [ ]:
if RUN_TRAINING:
    import random

    for example in random.Random(0).sample(list(dataset["validation"]), 3):
        messages = example["messages"]
        question, reference = messages[-2]["content"], messages[-1]["content"]
        history = messages[1:-2]  # tours précédents éventuels, sans le prompt système
        print(f"❓ {question}\n\nRÉFÉRENCE :\n{reference}\n\nMODÈLE :\n{ask(question, history)}\n\n{'─' * 80}\n")

## 13. Recharger un adaptateur existant

Cette section est exécutée uniquement lorsque `RUN_TRAINING = False`. `ADAPTER_PATH` vaut par défaut `/content/drive/MyDrive/mitterrand-qlora`. Placer les fichiers de l’adaptateur directement dans le dossier `mitterrand-qlora` de Mon Drive, puis exécuter les cellules. Cette section monte Google Drive automatiquement. Un autre dossier local ou un identifiant Hugging Face restent possibles. Après un entraînement, le modèle déjà en mémoire est utilisé directement.

Le prompt sauvegardé avec l'adaptateur est prioritaire. Pour un ancien adaptateur sans prompt, on reprend `data/system_prompt.txt` du dépôt, comme dans l'ancien notebook de discussion.


In [ ]:
if not RUN_TRAINING:
    import os

    if ADAPTER_PATH.startswith("/content/drive"):
        from google.colab import drive

        drive.mount("/content/drive")

    adapter_path = Path(ADAPTER_PATH).expanduser()
    local_adapter = adapter_path if adapter_path.is_absolute() else REPO_DIR / adapter_path
    if local_adapter.is_dir():
        adapter_dir = str(local_adapter.resolve())
    elif adapter_path.is_absolute() or ADAPTER_PATH == "adapter" or ADAPTER_PATH.startswith(("./", "../")):
        raise FileNotFoundError(
            f"Adaptateur local introuvable : {local_adapter}. "
            "Placez les fichiers de l’adaptateur dans ce dossier Drive, ou corrigez ADAPTER_PATH."
        )
    else:
        from huggingface_hub import snapshot_download

        adapter_dir = snapshot_download(ADAPTER_PATH)

    required_files = ("adapter_config.json", "adapter_model.safetensors")
    missing = [name for name in required_files if not (Path(adapter_dir) / name).is_file()]
    assert not missing, f"Adaptateur incomplet dans {adapter_dir} : {missing}"

    import json

    adapter_config = json.loads((Path(adapter_dir) / "adapter_config.json").read_text(encoding="utf-8"))
    assert adapter_config["base_model_name_or_path"] == MODEL_ID, (
        f"Cet adaptateur attend {adapter_config['base_model_name_or_path']}, mais MODEL_ID vaut {MODEL_ID}."
    )

    # Copie de data/system_prompt.txt, le prompt utilisé à l'entraînement. Sert pour les adaptateurs entraînés avec
    # une version du notebook qui n'enregistrait pas encore system_prompt.txt à côté de l'adaptateur.
    DEFAULT_SYSTEM_PROMPT = (REPO_DIR / "data" / "system_prompt.txt").read_text(encoding="utf-8").strip()

    system_prompt_file = os.path.join(adapter_dir, "system_prompt.txt")
    if os.path.exists(system_prompt_file):
        with open(system_prompt_file, encoding="utf-8") as f:
            SYSTEM_PROMPT = f.read().strip()
        source = "system_prompt.txt de l'adaptateur"
    else:
        SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT
        source = "prompt du projet (l'adaptateur n'en contient pas)"
    print(f"Adaptateur : {adapter_dir}\nFichiers : {sorted(os.listdir(adapter_dir))}\n\nPrompt système ({source}) :\n{SYSTEM_PROMPT}")

In [ ]:
if not RUN_TRAINING:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    base = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype=torch_dtype,
        device_map="auto",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype,
        ),
    )
    model = PeftModel.from_pretrained(base, adapter_dir)  # chargé en mode inférence
    # Utiliser le tokenizer et le chat template sauvegardés lors de l'entraînement.
    processor_source = adapter_dir if all(
        (Path(adapter_dir) / name).is_file() for name in ("processor_config.json", "tokenizer.json", "chat_template.jinja")
    ) else MODEL_ID
    processor = AutoProcessor.from_pretrained(processor_source)
    print(f"Modèle prêt. Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.1f} Go")

### Une question rapide


In [ ]:
print(ask("Monsieur Mitterrand, que représente pour vous la date du 10 mai 1981 ?"))

## 14. Interface Gradio

L'app de `app/app.py` appelle directement `ask()`, dans le même environnement que le modèle. Elle conserve son style, ses portraits, ses suggestions, son historique de cinq échanges et sa réinitialisation.

Dans Colab, l'interface s'affiche sous la cellule via le proxy interne (`share=False`). Utiliser Chrome ou Firefox. Depuis VS Code avec un serveur Colab, ouvrir le notebook dans Colab pour afficher Gradio.


In [ ]:
import inspect
from app.app import launch_app

assert "respond" in inspect.signature(launch_app).parameters, (
    f"L'app de {REPO_DIR} ne sait pas encore appeler le modèle : choisissez la branche qui contient launch_app(respond=...)."
)
launch_app(respond=ask)


## 15. (Optionnel) Fusion et publication

La fusion donne un modèle complet, utilisable sans PEFT (par exemple avec vLLM ou Ollama). Elle recharge Gemma 4 12B **en bfloat16, non quantifié**, ce qui demande environ 27 Go de RAM : il faut un environnement « RAM élevée » (A100).

Pour libérer la mémoire du GPU, redémarrez l'environnement avant la fusion, puis réexécutez les sections 1 à 3 (installation, configuration et connexion). L'adaptateur est sur le disque dans `OUTPUT_DIR` : il n'est pas perdu. La publication envoie sur un dépôt Hugging Face privé l'adaptateur, ou le modèle fusionné, avec son `system_prompt.txt`. Le parcours de rechargement de ce notebook peut ensuite le charger depuis ce dépôt.

In [ ]:
import os
import shutil

import torch
from huggingface_hub import create_repo, upload_folder
from transformers import AutoModelForMultimodalLM, AutoProcessor

MERGE = False  # @param {type:"boolean"}
PUSH_TO_HUB = False  # @param {type:"boolean"}
HUB_REPO_ID = "votre-compte/gemma-4-12b-mitterrand"  # @param {type:"string"}

if MERGE:
    from peft import PeftModel

    base = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, low_cpu_mem_usage=True)
    merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
    merged.save_pretrained("merged_model", safe_serialization=True, max_shard_size="2GB")
    AutoProcessor.from_pretrained(MODEL_ID).save_pretrained("merged_model")
    shutil.copy(os.path.join(OUTPUT_DIR, "system_prompt.txt"), "merged_model")
    print("Modèle fusionné sauvegardé dans merged_model/")

if PUSH_TO_HUB:
    folder = "merged_model" if MERGE else OUTPUT_DIR
    create_repo(HUB_REPO_ID, private=True, exist_ok=True)
    upload_folder(folder_path=folder, repo_id=HUB_REPO_ID, ignore_patterns=["checkpoint-*"])
    print(f"{'Modèle fusionné' if MERGE else 'Adaptateur'} publié en privé sur https://huggingface.co/{HUB_REPO_ID}")

## Retrouver le modèle plus tard

Conserver les fichiers de l’adaptateur directement dans `Mon Drive/mitterrand-qlora/`, sans sous-dossier `adapter` intermédiaire. Le dossier doit contenir au minimum `adapter_config.json` et `adapter_model.safetensors`. Conserver aussi les fichiers du tokenizer, du processor et `chat_template.jinja` pour utiliser le format sauvegardé.

À chaque session Colab, rouvrir `agent.ipynb`, garder `RUN_TRAINING = False` et `ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"`, puis exécuter les cellules dans l’ordre. Autoriser le montage de Drive lorsque Colab le demande. Aucun réentraînement ni copie de l’adaptateur dans le dépôt n’est nécessaire.

Les nouveaux entraînements sont sauvegardés au même emplacement via `DRIVE_DIR` lorsque `SAVE_TO_DRIVE = True`.
